# MedQuAD QLoRA training
This notebook only orchestrates the Python scripts staged in the private Kaggle dataset. Enable a GPU and internet in Kaggle settings.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import tarfile, os, sys
source = Path('/kaggle/input/llama-finetuning-code/project.tar')
project = Path('/kaggle/working/medical-project')
if not source.is_file():
    raise FileNotFoundError(f'Staged code missing at {source}')
project.mkdir(parents=True, exist_ok=True)
with tarfile.open(source) as archive:
    archive.extractall(project)
os.chdir(project)
sys.path.insert(0, str(project))
print('Project:', project)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
def run(module):
    subprocess.run([sys.executable, '-m', module], cwd=project, check=True)
run('src.prepare_data')


In [ ]:
run('src.train')
run('src.analyze_training')


In [ ]:
run('src.evaluate')
run('src.report')


In [ ]:
import yaml
config = yaml.safe_load((project / 'configs/config.yaml').read_text())
if config['publishing']['enabled']:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    run('src.upload_adapter')
else:
    print('Publishing disabled; adapter retained as Kaggle output')
